In [ ]:
"""
AQI Prediction Models for Cloud-Connected AQI Network

This script:
- Loads an AQI dataset (Datetime, PM_2.5, PM_1, SO2, CO, Total_AQI)
- Visualises pollutant and AQI trends over time
- Computes Kendall correlation between pollutants and Total_AQI
- Trains and evaluates:
    * Linear Regression (with standardisation)
    * Random Forest Regressor
    * Gradient Boosting Regressor
    * CatBoost Regressor (with Randomised hyperparameter search)
- Summarises model performance (Train R2, Test R2, RMSE, Train time)
"""

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

from catboost import CatBoostRegressor

sns.set_style("whitegrid")

# Path to your dataset inside the repo
DATA_PATH = "data/raw/aqi_dataset.csv"  # update if needed


# ---------------------------------------------------------------------------
# Data loading and basic inspection
# ---------------------------------------------------------------------------

def load_dataset(path: str) -> pd.DataFrame:
    """Load AQI dataset from CSV and parse Datetime column."""
    df = pd.read_csv(path, parse_dates=["Datetime"])
    print("Dataset loaded:", path)
    print("Shape:", df.shape)
    print("\nColumns:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    return df


# ---------------------------------------------------------------------------
# Visualisation helpers
# ---------------------------------------------------------------------------

def plot_time_series(df: pd.DataFrame) -> None:
    """Plot pollutants and Total_AQI over time."""
    fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

    for ax, col in zip(axes, ["PM_2.5", "PM_1", "SO2", "CO", "Total_AQI"]):
        ax.plot(df["Datetime"], df[col], linewidth=0.8)
        ax.set_ylabel(col)
        ax.set_title(f"{col} over time")

    axes[-1].set_xlabel("Datetime")
    plt.tight_layout()
    plt.show()


def plot_correlation_heatmap(df: pd.DataFrame) -> pd.DataFrame:
    """Compute and plot Kendall correlation between pollutants and Total_AQI."""
    corr_cols = ["PM_2.5", "PM_1", "SO2", "CO", "Total_AQI"]
    corr_matrix = df[corr_cols].corr(method="kendall")

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        corr_matrix,
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        vmin=-1,
        vmax=1,
    )
    plt.title("Kendall Correlation Heatmap: Pollutants vs Total AQI")
    plt.tight_layout()
    plt.show()

    print("\nCorrelation matrix:")
    print(corr_matrix)
    return corr_matrix


# ---------------------------------------------------------------------------
# Model training functions
# ---------------------------------------------------------------------------

def train_linear_regression(X_train, X_test, Y_train, Y_test):
    """Train and evaluate Linear Regression with standardised features."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    start = time.process_time()
    model = LinearRegression()
    model.fit(X_train_scaled, Y_train)
    train_time = time.process_time() - start

    y_pred = model.predict(X_test_scaled)

    train_score = model.score(X_train_scaled, Y_train)
    test_score = model.score(X_test_scaled, Y_test)
    rmse = np.sqrt(mean_squared_error(Y_test, y_pred))

    print(
        f"Linear Regression | Train R2: {train_score:.4f} | "
        f"Test R2: {test_score:.4f} | RMSE: {rmse:.4f} | "
        f"Train time: {train_time:.4f}s"
    )

    return {
        "name": "Linear Regression",
        "model": model,
        "train_r2": train_score,
        "test_r2": test_score,
        "rmse": rmse,
        "train_time": train_time,
    }


def train_random_forest(X_train, X_test, Y_train, Y_test):
    """Train and evaluate Random Forest Regressor."""
    model = RandomForestRegressor(
        n_estimators=100,
        min_samples_split=6,
        min_samples_leaf=1,
        max_features="log2",
        max_depth=100,
        bootstrap=True,
        random_state=42,
    )

    start = time.process_time()
    model.fit(X_train, Y_train)
    train_time = time.process_time() - start

    y_pred = model.predict(X_test)

    train_score = model.score(X_train, Y_train)
    test_score = model.score(X_test, Y_test)
    rmse = np.sqrt(mean_squared_error(Y_test, y_pred))

    print(
        f"Random Forest | Train R2: {train_score:.4f} | "
        f"Test R2: {test_score:.4f} | RMSE: {rmse:.4f} | "
        f"Train time: {train_time:.4f}s"
    )

    return {
        "name": "Random Forest",
        "model": model,
        "train_r2": train_score,
        "test_r2": test_score,
        "rmse": rmse,
        "train_time": train_time,
    }


def train_gradient_boosting(X_train, X_test, Y_train, Y_test):
    """Train and evaluate Gradient Boosting Regressor."""
    model = GradientBoostingRegressor(
        n_estimators=50,
        max_depth=10,
        learning_rate=0.1,
        max_features="sqrt",
        min_samples_split=2,
        min_samples_leaf=2,
        random_state=42,
    )

    start = time.process_time()
    model.fit(X_train, Y_train)
    train_time = time.process_time() - start

    y_pred = model.predict(X_test)

    train_score = model.score(X_train, Y_train)
    test_score = model.score(X_test, Y_test)
    rmse = np.sqrt(mean_squared_error(Y_test, y_pred))

    print(
        f"Gradient Boost | Train R2: {train_score:.4f} | "
        f"Test R2: {test_score:.4f} | RMSE: {rmse:.4f} | "
        f"Train time: {train_time:.4f}s"
    )

    return {
        "name": "Gradient Boosting",
        "model": model,
        "train_r2": train_score,
        "test_r2": test_score,
        "rmse": rmse,
        "train_time": train_time,
    }


def train_catboost_random_search(X_train, X_test, Y_train, Y_test):
    """Train CatBoost Regressor with Randomised hyperparameter search."""
    param_distributions = {
        "iterations": [200, 400, 600, 800],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "depth": [4, 6, 8, 10],
        "l2_leaf_reg": [1, 3, 5, 7, 9],
        "border_count": [32, 64, 128],
    }

    base_model = CatBoostRegressor(random_seed=42, silent=True)

    start = time.process_time()
    search = RandomizedSearchCV(
        base_model,
        param_distributions=param_distributions,
        n_iter=100,
        cv=5,
        n_jobs=-1,
        random_state=42,
    )
    search.fit(X_train, Y_train)
    train_time = time.process_time() - start

    best_model = search.best_estimator_
    print("Best CatBoost params:", search.best_params_)

    y_pred = best_model.predict(X_test)

    train_score = best_model.score(X_train, Y_train)
    test_score = best_model.score(X_test, Y_test)
    rmse = np.sqrt(mean_squared_error(Y_test, y_pred))

    print(
        f"CatBoost | Train R2: {train_score:.4f} | "
        f"Test R2: {test_score:.4f} | RMSE: {rmse:.4f} | "
        f"Train time: {train_time:.4f}s"
    )

    return {
        "name": "CatBoost Regressor",
        "model": best_model,
        "train_r2": train_score,
        "test_r2": test_score,
        "rmse": rmse,
        "train_time": train_time,
    }


# ---------------------------------------------------------------------------
# Results summary and plotting
# ---------------------------------------------------------------------------

def summarise_results(results_list):
    """Create a DataFrame summarising model performance."""
    results = pd.DataFrame({
        "Model": [r["name"] for r in results_list],
        "Train R2": [r["train_r2"] for r in results_list],
        "Test R2": [r["test_r2"] for r in results_list],
        "RMSE": [r["rmse"] for r in results_list],
        "Train Time (s)": [r["train_time"] for r in results_list],
    })

    results.sort_values("Test R2", ascending=False, inplace=True)
    results.reset_index(drop=True, inplace=True)

    print("\nModel performance summary:")
    print(results)

    return results


def plot_model_comparison(results: pd.DataFrame) -> None:
    """Plot Test R2 and RMSE for each model."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(results["Model"], results["Test R2"], color="steelblue")
    axes[0].set_title("Test R2 Score by Model")
    axes[0].set_ylabel("R2 Score")
    axes[0].tick_params(axis="x", rotation=20)

    axes[1].bar(results["Model"], results["RMSE"], color="indianred")
    axes[1].set_title("RMSE by Model")
    axes[1].set_ylabel("RMSE")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def main():
    # Load data
    df = load_dataset(DATA_PATH)

    # Quick preview
    print("\nFirst rows:")
    print(df.head())
    print("\nLast rows:")
    print(df.tail())

    # Visualisations
    plot_time_series(df)
    plot_correlation_heatmap(df)

    # Feature and target definition
    X = df[["PM_2.5", "PM_1", "SO2", "CO"]]
    Y = df["Total_AQI"]

    # Train/test split
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )
    print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

    # Train models
    results_list = []
    results_list.append(train_linear_regression(X_train, X_test, Y_train, Y_test))
    results_list.append(train_random_forest(X_train, X_test, Y_train, Y_test))
    results_list.append(train_gradient_boosting(X_train, X_test, Y_train, Y_test))
    results_list.append(train_catboost_random_search(X_train, X_test, Y_train, Y_test))

    # Summarise and plot
    results_df = summarise_results(results_list)
    plot_model_comparison(results_df)


if __name__ == "__main__":
    main()